In [ ]:
import glob

import pandas as pd

# FIXME: replace with dir to ISRUC data/annotations
f_path = "<path-to-ISRUC>"
xlsx_paths = glob.glob(f"{f_path}/**/*.xlsx", recursive=True)

s_id_to_l_events_map = {}
for xp in xlsx_paths:
    ds_idx = xp.find("isruc-sg")
    ds = xp[ds_idx:ds_idx + len("isruc-sg") + 1]
    s_id = xp[ds_idx + len(ds) + 1:].replace("/", "#")
    print(f"Reading {s_id} from {ds}")

    s_id_to_l_events_map[f"{ds}#{s_id}"] = {"l_out": [], "l_on": []}

    data = pd.read_excel(xp)
    if "Epoch" not in data.columns:
        print(f"WARNING: Likely no header in {xp}")
        data = pd.read_excel(xp, header=None)
        print(f"First row: {data.iloc[0].to_list()}")
        print(f"Total len: {max(data.iloc[:, 0])}")
    else:
        print(f"Total len: {max(data['Epoch'])}")

    # search for "L out" and "L on" in the data
    for i, row in data.iterrows():
        if "l out" in str(row).lower() or "lights out" in str(row).lower() or "l off" in str(
                row).lower() or "lights off" in str(row).lower():
            s_id_to_l_events_map[f"{ds}#{s_id}"]["l_out"].append(i)
        if "l on" in str(row).lower() or "lights on" in str(row).lower():
            s_id_to_l_events_map[f"{ds}#{s_id}"]["l_on"].append(i)

In [ ]:
# save to json
import json

with open("isruc_lights_events.json", "w") as f:
    json.dump(s_id_to_l_events_map, f, indent=None)